# Chapter 7 — Bayesian Generalization

Companion notebook for the [probintro textbook](https://josephausterweil.github.io/probintro/intro2/07_generalization/).

When Chibany sees a few examples of a concept, which *new* things should they expect to belong too? This notebook builds the **number game** and watches the **size principle** and **strong sampling** produce a generalization gradient — all by enumerating hypotheses and applying Bayes' rule.

## Setup

This notebook uses only JAX (no GenJAX needed).

In [ ]:
!pip install jax -q
import jax
import jax.numpy as jnp
print('✅ ready')

## 1. Warm-up: a concept is a *set* of hypotheses

A tiny 4-bento world with two candidate rules. We observe one sticker'd bento and update our belief over the rules, then predict which other bentos get the sticker.

In [ ]:
import jax.numpy as jnp

# A tiny menu of 4 bentos (columns 0-3). Two candidate rules (rows).
rule_A = jnp.array([1.0, 1.0, 0.0, 0.0])   # Rule A = {0, 1}    ("the two lightest")
rule_B = jnp.array([1.0, 1.0, 1.0, 0.0])   # Rule B = {0, 1, 2} ("the lighter half")
prior_A = prior_B = 0.5

# Observed: bento 1 has the sticker. Likelihood = is bento 1 in the rule?
post_A = prior_A * rule_A[1]
post_B = prior_B * rule_B[1]
total = post_A + post_B
post_A, post_B = post_A/total, post_B/total
print("posterior:  Rule A =", float(post_A), " Rule B =", float(post_B))

# Predict: P(sticker | data) for each bento = total posterior on rules that contain it
for bento in range(4):
    p = post_A * rule_A[bento] + post_B * rule_B[bento]
    print(f"  predicted P(sticker) for bento {bento} = {float(p)}")

## 2. The number game

The universe is the numbers 1–30. Seven candidate rules (hypotheses), each a row of 0/1 membership. A uniform prior over rules to start.

In [ ]:
numbers = list(range(1, 31))
def make_rule(test):
    return jnp.array([1.0 if test(n) else 0.0 for n in numbers])

labels = ["all numbers", "even numbers", "multiples of 3", "multiples of 10",
          "powers of 2", "numbers 1-10", "numbers 20-30"]
H = jnp.stack([
    make_rule(lambda n: True),                  # all numbers   (size 30)
    make_rule(lambda n: n % 2 == 0),            # even          (size 15)
    make_rule(lambda n: n % 3 == 0),            # multiples of 3(size 10)
    make_rule(lambda n: n % 10 == 0),           # multiples of 10(size 3)
    make_rule(lambda n: n in (1, 2, 4, 8, 16)), # powers of 2   (size 5)
    make_rule(lambda n: 1 <= n <= 10),          # numbers 1-10  (size 10)
    make_rule(lambda n: 20 <= n <= 30),         # numbers 20-30 (size 11)
])
prior = jnp.ones(H.shape[0]) / H.shape[0]

sizes = jax.vmap(jnp.sum)(H)
for label, size in zip(labels, sizes):
    print(f"|h| for {label:16s} = {int(size)}")

## 3. The size principle (weak vs. strong sampling)

**Weak sampling**: an example is just *consistent* or not. **Strong sampling**: the example was *deliberately drawn from the concept*, so a smaller rule (which had fewer numbers to choose from) assigns it higher probability — $1/|h|$. The size principle: among consistent rules, the *smallest* wins, and more examples sharpen the effect.

In [ ]:
def log_likelihood(rule, observed, strong):
    in_rule = rule[observed]
    size = rule.sum()
    if strong:
        per = jnp.where(in_rule > 0, -jnp.log(size), -jnp.inf)   # 1/|h| per example
    else:
        per = jnp.where(in_rule > 0, 0.0, -jnp.inf)              # consistent or not
    return per.sum()

def posterior(observed, strong):
    log_post = jnp.log(prior) + jax.vmap(lambda r: log_likelihood(r, observed, strong))(H)
    log_post = log_post - log_post.max()
    p = jnp.exp(log_post)
    return p / p.sum()

one   = jnp.array([9])            # the number 10 (column n-1)
three = jnp.array([9, 19, 29])    # 10, 20, 30

print("STRONG, after {10}:")
for l, p in zip(labels, posterior(one, strong=True)):
    print(f"   {l:16s} {round(float(p), 3)}")
print("\nSTRONG, after {10, 20, 30}:")
for l, p in zip(labels, posterior(three, strong=True)):
    print(f"   {l:16s} {round(float(p), 3)}")

After just `{10}`, *multiples of 10* already leads (it's the smallest consistent rule). After `{10, 20, 30}` it shoots to near-certainty — that's the size principle compounding with more data.

## 4. The generalization gradient

Predict $P(n \text{ has the property} \mid \text{data})$ for every number, by summing the posterior over the rules that contain $n$.

In [ ]:
post = posterior(one, strong=True)
predictive = jnp.array([jnp.sum(post * H[:, n - 1]) for n in numbers])

for n in [10, 2, 5, 7, 25]:
    print(f"  P({n:2d} has the property | saw 10) = {round(float(predictive[n - 1]), 3)}")

## Your turn

1. Compare **weak** vs **strong** sampling: call `posterior(one, strong=False)` and see how the size principle disappears (consistent rules are no longer penalized for being large).
2. Add a new hypothesis to `H` (e.g. "square numbers") and re-run — how does it compete?
3. Plot the full generalization gradient `predictive` over 1–30 with `matplotlib` and watch it concentrate as you go from `{10}` to `{10,20,30}`.
4. (Continuous version) See the textbook's *Continuous Concepts & Shepard's Law* section for the interval-hypothesis version that yields Shepard's exponential generalization gradient.